# RAG Chat Main — Naive Combined (Dukcapil + OPD)

Naive RAG pipeline yang query **DUA vector store sekaligus** dalam satu pass, **tanpa router**.

**Posisi di project:**
- `rag_chat.ipynb` → dukcapil only (4 variants, sudah ada)
- `rag_chat_opd.ipynb` → OPD only (3 variants, sudah ada)
- **`rag_chat_main.ipynb` ← INI: combined, no routing**
- `agenticrag/2-agentic_router.ipynb` → agentic dgn router (sudah ada)

**Strategi:**
- Retrieve hybrid (BM25 + dense, RRF, weights 0.5/0.5) dari **masing-masing store**
- `k_per_store=4` → total 8 docs (match agentic-both: 4+4 = 8)
- Tag tiap doc dgn `metadata._source` (`dukcapil` atau `opd`)
- Generate pakai `PROMPT_COMBINED` yang generic (tidak force struktur 2-bagian)

**Hipotesis latency vs agentic:**
- ✅ Hilang router overhead (~2.4s LLM call)
- ❌ Selalu retrieve 2 store (boros buat query single-domain atau off-topic)
- ❌ Selalu generate dgn context 2x (~8 docs)
- **Net**: di dataset kecil (150+61 docs), kemungkinan naive combined MENANG karena router overhead lebih besar dari ekstra retrieve cost.

## Step 1 — Import module

In [1]:
from rag_chat_main import (
    ask_main,
    retrieve_combined,
    retrieve_dukcapil_hybrid,
    retrieve_opd_hybrid,
    PROMPT_COMBINED,
    format_context_combined,
    vs_dukcapil,
    vs_opd,
    dukcapil_docs_all,
    opd_docs_all,
    llm,
    embeddings,
)

print(f"✓ Module loaded")
print(f"   Dukcapil docs in-memory: {len(dukcapil_docs_all)}")
print(f"   OPD docs in-memory:      {len(opd_docs_all)}")

c:\Users\Nafisha\Documents\RAGTrial\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.
Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.


✓ Module loaded
   Dukcapil docs in-memory: 150
   OPD docs in-memory:      61


## Step 2 — Inspect retrieval (sanity check)

Cek dulu sebelum generate: untuk query single-domain, apakah retrieval dari store yang TIDAK relevan menghasilkan docs yang ke-mix (noisy)? Ini penting untuk evaluasi kualitatif nanti.

In [2]:
inspect_queries = [
    ("Apa syarat KTP-el?", "dukcapil-only"),
    ("Alamat Dinas Pariwisata Batang?", "opd-only"),
    ("Cara bikin akta kelahiran dan ke dinas mana?", "both"),
]
for q, expected in inspect_queries:
    docs = retrieve_combined(q, k_per_store=4)
    print(f"\nQ: {q}  (expected: {expected})")
    for i, d in enumerate(docs, 1):
        src = d.metadata.get("_source")
        if src == "opd":
            label = f"OPD-{d.metadata.get('nama_opd', '?')[:40]}"
        else:
            label = f"DUK-{d.metadata.get('section', '?')[:30]} hal{d.metadata.get('page', '?')}"
        snippet = d.page_content[:80].replace("\n", " ")
        print(f"   [{i:2d}] {src:8s} | {label:50s} | {snippet}...")


Q: Apa syarat KTP-el?  (expected: dukcapil-only)
   [ 1] dukcapil | DUK-BAB II - Pertanyaan dan Jawaba hal?            | 2. Apakah Orang Asing boleh memiliki KTP-el? Jawaban: Berdasarkan Pasal 16 Perpr...
   [ 2] dukcapil | DUK-BAB II - Pertanyaan dan Jawaba hal?            | 5. Apa perbedaan antara KTP-el WNI dan KTP-el WNA? Jawaban: Perbedaan: a. KTP-el...
   [ 3] dukcapil | DUK-BAB II - Pertanyaan dan Jawaba hal?            | 1. Bagaimana penerbitan KTP-el pertama kali bagi WNI? Jawaban: Berdasarkan Pasal...
   [ 4] dukcapil | DUK-BAB II - Pertanyaan dan Jawaba hal?            | 3 Apa manfaat ketika telah mendaftar sebagai penduduk nonpermanen? Jawaban: Pend...
   [ 5] opd      | OPD-Kecamatan Kandeman                             | Nama OPD: Kecamatan Kandeman Tipe: Kecamatan Alamat: Jl. Raya Kandeman Email: ke...
   [ 6] opd      | OPD-RSUD Limpung                                   | Nama OPD: RSUD Limpung Tipe: RSUD Alamat: Jl. Dr. Sutomo No. 17 Limpung Email: r...
   [ 7] opd   

## Step 3 — Smoke test 6 query (end-to-end + latency)

Mix coverage: dukcapil-only, opd-only, both, dan satu out-of-scope.

In [3]:
TEST_QUERIES = [
    # dukcapil-only
    "Apa syarat penerbitan KTP-el pertama kali bagi WNI?",
    "Bagaimana prosedur pindah domisili untuk WNA pemegang KITAP?",
    # opd-only
    "Alamat Dinas Pariwisata Kabupaten Batang di mana?",
    "Nomor telepon Sekretariat Daerah Batang berapa?",
    # both
    "Mau urus akta kematian, ke dinas mana dan apa syaratnya?",
    # off-topic
    "Resep nasi goreng spesial?",
]

results = []
for q in TEST_QUERIES:
    r = ask_main(q, verbose=True)
    results.append(r)

Q: Apa syarat penerbitan KTP-el pertama kali bagi WNI?
   Docs: 4 dukcapil + 4 opd = 8 total
   Timing — retrieve: 0.80s | generate: 2.87s | TOTAL: 3.67s
   Answer: Syarat penerbitan KTP-el pertama kali bagi WNI adalah:
a. Telah berusia 17 tahun, sudah kawin atau pernah kawin; dan
b. Fotokopi KK.

[Sumber: Dukcapil 1: section=BAB II - Pertanyaan dan Jawaban]

Q: Bagaimana prosedur pindah domisili untuk WNA pemegang KITAP?
   Docs: 4 dukcapil + 4 opd = 8 total
   Timing — retrieve: 1.13s | generate: 5.21s | TOTAL: 6.35s
   Answer: Prosedur pindah domisili untuk WNA pemegang KITAP berbeda tergantung tujuan pindahnya:

**1. Pindah dalam 1 (satu) kabupaten/kota (berbeda kel

Q: Alamat Dinas Pariwisata Kabupaten Batang di mana?
   Docs: 4 dukcapil + 4 opd = 8 total
   Timing — retrieve: 0.90s | generate: 2.28s | TOTAL: 3.18s
   Answer: Alamat Dinas Pariwisata Kabupaten Batang adalah Jl. Urip Sumoharjo No. 34 Batang.
[Sumber: Dinas Pariwisata, Kepemudaan dan Olahraga, nomor 12]

Q: Nomor tel

## Step 4 — Latency summary

In [4]:
import pandas as pd

summary = pd.DataFrame([
    {
        "query": r["question"][:55] + ("..." if len(r["question"]) > 55 else ""),
        "retrieve_s": round(r["timings"]["retrieve"], 2),
        "generate_s": round(r["timings"]["generate"], 2),
        "total_s":    round(r["timings"]["total"], 2),
    }
    for r in results
])
print(summary.to_string(index=False))
print(f"\nMean total: {summary['total_s'].mean():.2f}s")
print(f"Mean retrieve: {summary['retrieve_s'].mean():.2f}s")
print(f"Mean generate: {summary['generate_s'].mean():.2f}s")

                                                     query  retrieve_s  generate_s  total_s
       Apa syarat penerbitan KTP-el pertama kali bagi WNI?        0.80        2.87     3.67
Bagaimana prosedur pindah domisili untuk WNA pemegang K...        1.13        5.21     6.35
         Alamat Dinas Pariwisata Kabupaten Batang di mana?        0.90        2.28     3.18
           Nomor telepon Sekretariat Daerah Batang berapa?        0.80        6.88     7.68
Mau urus akta kematian, ke dinas mana dan apa syaratnya...        0.95       12.61    13.55
                                Resep nasi goreng spesial?        0.90        6.25     7.15

Mean total: 6.93s
Mean retrieve: 0.91s
Mean generate: 6.02s


## Step 5 — Observasi

Yang perlu dicatat manual setelah run:
1. **Apakah retrieval ke-mix?** Untuk query dukcapil-only (mis. KTP-el), apakah ada docs OPD yang ikut top-4? Kalau ya, apakah mengganggu jawaban?
2. **Generate behavior off-topic**: untuk "resep nasi goreng", apakah model konsisten jawab "informasi tidak ditemukan" sesuai aturan 7 di `PROMPT_COMBINED`?
3. **Latency vs agentic**: bandingin mean total di sini vs ~7.2s agentic (dari notebook 3 sebelumnya). Hipotesis menang/kalah?

Comparison head-to-head lengkap → lihat `agenticrag/3-compare_agentic_vs_naive.ipynb`.